In [13]:
import pandas as pd
from groq import Groq
from dotenv import load_dotenv
import os

load_dotenv()
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
client = Groq(api_key=GROQ_API_KEY)

print("AI model ready!")

AI model ready!


In [14]:
rfm = pd.read_csv("../data/processed/rfm_segments.csv")
forecast = pd.read_csv("../data/processed/demand_forecast.csv")
anomalies = pd.read_csv("../data/processed/seller_anomalies.csv")
anomalies = anomalies[anomalies['anomaly'] == -1]

# Build summary stats
total_customers   = len(rfm)
champions         = (rfm['Segment'] == 'Champions').sum()
at_risk           = (rfm['Segment'] == 'At Risk').sum()
churned           = rfm['Churned'].sum()
anomaly_count     = len(anomalies)
worst_delay       = anomalies['avg_delay'].max()
next_30_day_avg   = forecast.tail(30)['yhat'].mean()

print(f"Customers: {total_customers:,}")
print(f"Champions: {champions:,}")
print(f"At Risk: {at_risk:,}")
print(f"Anomalies: {anomaly_count:,}")
print(f"Forecast next 30 days avg: {next_30_day_avg:.0f} orders/day")

Customers: 96,478
Champions: 7,781
At Risk: 39,822
Anomalies: 149
Forecast next 30 days avg: 301 orders/day


In [15]:



report_prompt = f"""
You are an Operations Intelligence AI for an e-commerce company.
Generate a professional weekly operations report based on this data:

CUSTOMER ANALYTICS:
- Total customers: {total_customers:,}
- Champions (best customers): {champions:,}
- At Risk customers: {at_risk:,}
- Churned customers: {churned:,}

DEMAND FORECAST:
- Expected orders next 30 days: {next_30_day_avg:.0f} per day

ANOMALY DETECTION:
- Anomalous sellers flagged: {anomaly_count:,}
- Worst delivery delay detected: {worst_delay:.1f} days

Write a 3-paragraph executive summary with:
1. Overall business health
2. Key risks and alerts
3. Recommended actions for operations team
"""

response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",  # Updated to a supported, active model
    messages=[{"role": "user", "content": report_prompt}]
)
print(response.choices[0].message.content)

**Executive Summary**

The current state of our e-commerce operations indicates a mixed bag in terms of overall business health. On one hand, we have a substantial total customer base of 96,478, with 7,781 champions who are our most loyal and valuable customers. However, a deeper dive into the numbers reveals concerns, particularly with a significant portion of our customer base (41.3%) categorized as "At Risk" and a staggering 68,837 customers having already churned. This dynamic warrants careful attention to improve customer retention and overall customer lifecycle management.

Key risks and alerts have been identified through our anomaly detection systems and customer analytics. Notably, 149 anomalous sellers have been flagged, which may indicate potential issues with product quality, pricing, or seller behavior that could negatively impact customer satisfaction and trust in our platform. Furthermore, the worst delivery delay detected stands at 35.0 days, which is significantly abov

In [16]:

email_prompt = f"""
You are an AI assistant for an e-commerce operations team.
Write a professional supplier reorder email based on this forecast:

- Product category: bed_bath_table (highest demand category)
- Current avg daily orders: {next_30_day_avg:.0f}
- Forecast shows demand will increase next 30 days
- Current stock risk: Medium

Write a concise, professional reorder email to the supplier.
Include: subject line, greeting, order details, urgency, closing.
"""

email_response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[{"role": "user", "content": email_prompt}]
)
print("=== AI GENERATED SUPPLIER EMAIL ===\n")
print(email_response.choices[0].message.content)

=== AI GENERATED SUPPLIER EMAIL ===

Subject: Urgent: Reorder for Bed, Bath, and Table Category to Meet Increasing Demand

Dear Valued Supplier,

We are writing to request a prompt reorder for our bed, bath, and table category products, which are currently experiencing high demand. According to our latest forecast, we anticipate a significant increase in orders over the next 30 days, with our current average daily orders already at 301.

In light of this projected growth, we are facing a medium stock risk and would like to replenish our inventory to ensure we can continue to meet customer demand. To avoid stockouts and potential delays, we kindly request that you expedite the following reorder:

* Product Category: Bed, Bath, and Table
* Estimated Quantity: [Insert quantity based on forecast and current stock levels]
* Desired Delivery Date: [Insert date, ideally within the next 14-21 days]

We would appreciate it if you could confirm receipt of this request and provide an estimated sh

In [17]:
with open('../outputs/weekly_report.txt', 'w') as f:
    f.write("=== SMARTOPS WEEKLY OPS REPORT ===\n\n")
    f.write(response.choices[0].message.content)

with open('../outputs/supplier_email.txt', 'w') as f:
    f.write("=== AI GENERATED SUPPLIER EMAIL ===\n\n")
    f.write(email_response.choices[0].message.content)

print("Module 4 Complete!")
print("Reports saved to outputs/ folder!")

Module 4 Complete!
Reports saved to outputs/ folder!


In [18]:
pip install secure-smtplib

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [19]:
import os

# Get project root folder
BASE_DIR = os.path.dirname(os.path.dirname(os.path.abspath('__file__')))
outputs_dir = os.path.join(BASE_DIR, "outputs")
os.makedirs(outputs_dir, exist_ok=True)

# Save weekly report
with open(os.path.join(outputs_dir, "weekly_report.txt"),
          "w", encoding="utf-8") as f:
    f.write(response.choices[0].message.content)

# Save supplier email
with open(os.path.join(outputs_dir, "supplier_email.txt"),
          "w", encoding="utf-8") as f:
    f.write(email_response.choices[0].message.content)

print("Files saved to:", outputs_dir)
print("Weekly report length:", 
      len(response.choices[0].message.content))
print("Supplier email length:", 
      len(email_response.choices[0].message.content))

Files saved to: c:\Users\abmin\AI-Powered-E-Commerce-Operations-Intelligence-System\outputs
Weekly report length: 2158
Supplier email length: 1235


In [20]:
import os
from dotenv import load_dotenv

load_dotenv()

print("BREVO_LOGIN =", os.getenv("BREVO_LOGIN"))
print("SENDER_EMAIL =", os.getenv("SENDER_EMAIL"))
print("BREVO_SMTP_KEY =", os.getenv("BREVO_SMTP_KEY"))

BREVO_LOGIN = None
SENDER_EMAIL = None
BREVO_SMTP_KEY = None


In [21]:
import os
from dotenv import load_dotenv
load_dotenv()
import smtplib
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart

def send_report_email(report_text, recipient_email):
    smtp_server     = "smtp-relay.brevo.com"
    port            = 587
    login_email     = os.getenv("BREVO_LOGIN")
    sender_email    = os.getenv("SENDER_EMAIL")
    sender_password = os.getenv("BREVO_SMTP_KEY")
    
    msg = MIMEMultipart()
    msg['From']    = sender_email
    msg['To']      = recipient_email
    msg['Subject'] = "SmartOps Weekly Operations Report"
    
    msg.attach(MIMEText(report_text, 'plain'))
    
    with smtplib.SMTP(smtp_server, port) as server:
        server.starttls()
        server.login(login_email, sender_password)
        server.sendmail(sender_email,
                       recipient_email,
                       msg.as_string())
    
    print(f"Report sent to {recipient_email}")
    
# 1. Grab both raw text outputs from your variables
summary_text = response.choices[0].message.content
supplier_text = email_response.choices[0].message.content

# 2. Extract the Supplier Email body and drop its original subject line
clean_supplier = supplier_text.replace("Subject: Urgent Reorder Request for Bed, Bath, and Table Products\n\n", "")

# 3. Cut off the supplier text right before "Best regards," 
# This completely drops the middle signature block so it won't clutter the center
supplier_body_only = clean_supplier.partition("Best regards,")[0].strip()

# 4. Define your active local or hosted Streamlit dashboard link
# Change "localhost:8501" to your production URL when you host it online!
streamlit_dashboard_url = "https://your-app.streamlit.app"
clean_supplier = supplier_text.replace(
    "Subject: Urgent Reorder Request for Bed, Bath, and Table Products\n\n", "")
supplier_body_only = clean_supplier.partition(
    "Best regards,")[0].strip()

payload = f"""
======================================================================
                 OUTBOUND SUPPLIER REORDER DRAFT
======================================================================
{supplier_body_only}


======================================================================
                  INTERNAL EXECUTIVE SUMMARY 
======================================================================
{summary_text}

======================================================================
📊 LIVE SMARTOPS DASHBOARD CALL-TO-ACTION
======================================================================
To inspect real-time anomaly trends or update inventory parameters:
👉 CLICK HERE TO OPEN DASHBOARD: {streamlit_dashboard_url}
======================================================================

======================================================================
Best regards,
Vinay
E-commerce Operations Team
======================================================================
"""

# 5. Fire the perfectly rearranged single payload to your inbox
#send_report_email(payload, "vinaygod100@gmail.com")
#print("✅ Rearranged email sequence dispatched successfully!")

print("✅ AI Report Generated Successfully!")
print(payload)

✅ AI Report Generated Successfully!

                 OUTBOUND SUPPLIER REORDER DRAFT
Subject: Urgent: Reorder for Bed, Bath, and Table Category to Meet Increasing Demand

Dear Valued Supplier,

We are writing to request a prompt reorder for our bed, bath, and table category products, which are currently experiencing high demand. According to our latest forecast, we anticipate a significant increase in orders over the next 30 days, with our current average daily orders already at 301.

In light of this projected growth, we are facing a medium stock risk and would like to replenish our inventory to ensure we can continue to meet customer demand. To avoid stockouts and potential delays, we kindly request that you expedite the following reorder:

* Product Category: Bed, Bath, and Table
* Estimated Quantity: [Insert quantity based on forecast and current stock levels]
* Desired Delivery Date: [Insert date, ideally within the next 14-21 days]

We would appreciate it if you could confirm re